# Hybrid search: BM25+ with BGE-base (Colab GPU)

Scores BM25+ and BGE-base combined with reciprocal rank fusion (RRF), with and without a
cross-encoder reranker, using the project's `HybridRetriever` and `CrossEncoderReranker`.
Both searches cover all 511,958 documents; each question is searched once and every
variant reuses those results.

1. **Runtime → Change runtime type → T4 GPU, with High-RAM on.** The corpus text, the BM25+
   index and the BGE-base index are loaded together.
2. It reuses files on Google Drive in `rag-dense/`: `documents.parquet`, `questions.jsonl`,
   `indexes/bm25` and `dense_base_titled`. No new embedding is needed.
3. **Part 3** scores four variants on the 300 development questions (about 15–20 minutes).
   **Part 4** picks the best by development Recall@10. **Part 5** checks only that setting,
   and BM25+ alone, once on the 100 calibration questions. The final split is never used.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 1. Drive and project code

In [ ]:
import json
import os
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/rag-dense"
CORPUS = f"{WORK}/documents.parquet"
QUESTIONS = f"{WORK}/questions.jsonl"
DRIVE_BM25 = f"{WORK}/indexes/bm25"
DENSE = f"{WORK}/dense_base_titled"
BM25 = "/content/indexes/bm25"  # Local copy: much faster than reading from Drive.

BRANCH = "hira/dev"
if not os.path.exists("/content/project"):
    !git clone --depth 1 -b {BRANCH} https://github.com/hira-mian/Agentic-Enterprise-RAG-System.git /content/project
%cd /content/project
!git log -1 --format='%h %s'
!pip install -q -r requirements.txt

## 2. Check the indexes

Both indexes must come from the same `documents.parquet`. If the BM25+ index on Drive is
missing or was built from a different copy, it is rebuilt here (about 5 minutes).

In [ ]:
for path in (CORPUS, QUESTIONS, f"{DENSE}/manifest.json"):
    assert os.path.exists(path), f"Missing {path}"
dense_manifest = json.load(open(f"{DENSE}/manifest.json"))
assert dense_manifest.get("complete"), "The BGE-base index is incomplete"
print("Dense model:", dense_manifest["model"]["name"], dense_manifest["model"]["revision"])

def bm25_matches(path):
    manifest = f"{path}/manifest.json"
    return os.path.exists(manifest) and json.load(open(manifest))["corpus_sha256"] == dense_manifest["corpus_sha256"]

if not bm25_matches(BM25):
    if bm25_matches(DRIVE_BM25):
        print("Copying the BM25+ index from Drive...")
        !rm -rf {BM25} && mkdir -p /content/indexes && cp -r {DRIVE_BM25} {BM25}
    else:
        print("Building a BM25+ index from the same corpus (about 5 minutes)...")
        !python -m src.retrieval.corpus_bm25 --corpus {CORPUS} --output {BM25}
assert bm25_matches(BM25)
print("Indexes ready.")

## 3. Development: four variants

- RRF with equal weights (the `HybridRetriever` default) and with BM25+ weight 0.7.
- The better of those two, reranked by `bge-reranker-base`, and by `bge-reranker-v2-m3`.

Each search keeps the top 50 documents per method; the reranker rescores the 50 fused
candidates. Each variant is saved to `evaluation/results/hybrid_..._full_development/`.

In [ ]:
COMMON = f"--bm25-index {BM25} --dense-index {DENSE} --corpus {CORPUS} --questions {QUESTIONS} --device cuda"
!python -m src.evaluation.run_hybrid {COMMON} --split development \
    --bm25-weights 0.5,0.7 --rerankers bge-reranker-base,bge-reranker-v2-m3

## 4. Pick the best development setting

In [ ]:
import glob

runs = {}
for path in sorted(glob.glob("evaluation/results/hybrid_*_full_development/summary.json")):
    s = json.load(open(path))
    runs[s["method"]] = s
bm25_dev = json.load(open("evaluation/results/bm25_full_development/summary.json"))["overall"]
print(f"{'run':48} {'R@10':>6} {'nDCG@10':>8}")
print(f"{'BM25+ alone (baseline)':48} {bm25_dev['recall@10']['mean']:>6.3f} {bm25_dev['ndcg@10']['mean']:>8.3f}")
for name, s in runs.items():
    o = s["overall"]
    print(f"{name:48} {o['recall@10']['mean']:>6.3f} {o['ndcg@10']['mean']:>8.3f}")

# Highest Recall@10, then nDCG@10; on an exact tie prefer no reranker (cheaper).
best = max(runs.values(), key=lambda s: (s["overall"]["recall@10"]["mean"], s["overall"]["ndcg@10"]["mean"],
                                         s["index"]["reranker"] is None))
config = best["index"]["config"]
WEIGHT = config["bm25_weight"] / (config["bm25_weight"] + config["dense_weight"])
reranker = best["index"]["reranker"]
LABELS = {"BAAI/bge-reranker-base": "bge-reranker-base", "BAAI/bge-reranker-v2-m3": "bge-reranker-v2-m3"}
# Reuse the exact reranker revision scored on development.
RERANKER = f"{LABELS[reranker['name']]}@{reranker['revision']}" if reranker else ""
WEIGHT = f"{WEIGHT:g}"  # Text form for the command below.
print(f"\nChosen: {best['method']} (BM25+ weight {WEIGHT}, reranker: {RERANKER or 'none'})")

## 5. Calibration: one check of the chosen setting

Runs **only** the chosen setting, plus BM25+ alone and BGE-base alone, on the 100
calibration questions. Do not change settings based on this result.

In [ ]:
!python -m src.evaluation.run_hybrid {COMMON} --split calibration --write-baselines \
    --bm25-weights {WEIGHT} --rerankers "{RERANKER}" --rerank-bm25-weight {WEIGHT}

## 6. Save the results

Copies every result folder from this notebook to Drive (`rag-dense/results`) and downloads
`hybrid_results.zip`. Upload the zip to add the results to the repository.

In [ ]:
!mkdir -p {WORK}/results
!cd evaluation/results && cp -r hybrid_*_full_development *_full_calibration {WORK}/results/
!cd evaluation/results && zip -qr /content/hybrid_results.zip hybrid_*_full_development *_full_calibration
from google.colab import files
files.download("/content/hybrid_results.zip")